In [7]:
from pathlib import Path
from typing import Optional

import pandas as pd

from scripts.preprocess_wsi_tiles import (
    DATA_PATH,
    PROJECT_DATA_PATH,
    IMAGE_DST_PATH,
    TCGA_RAW_PATH,
    CPTAC_RAW_PATH,
    run_tcga_tiling,
    run_cptac_tiling,
    set_seed,
)

set_seed(42)

RESULT_PATH = Path("../../results")
VERIFY_RESULT_PATH = RESULT_PATH / "pancreatic_cancer_pathology" / "data_verification"
COMMON_CLINICAL_RESULT_PATH = VERIFY_RESULT_PATH / "common_data"

TARGET_MPP = 1.0
TILE_SIZE = 1024
TISSUE_THRESHOLD = 0.15

# Low-res tissue mask에서 먼저 흰 배경 tile을 제외한 뒤 실제 tile을 읽습니다.
PREFILTER_TISSUE_THRESHOLD = 0.01
TCGA_TISSUE_MASK_DOWNSAMPLE = 64

# 처음 테스트할 때는 1 또는 3으로 설정하고, 확인 후 None으로 바꾸세요.
DEBUG_MAX_CASES: Optional[int] = None

# case 단위 병렬 처리입니다. CPTAC는 DICOM frame decoding이 무거워 너무 크게 잡지 않습니다.
TCGA_MAX_WORKERS = 4
CPTAC_MAX_WORKERS = 2
SHOW_TILE_PROGRESS = False  # True는 max_workers=1 디버깅 때 가장 보기 좋습니다.

SKIP_EXISTING = True
OVERWRITE_METADATA = False

# PNG는 무손실이지만 느리고 용량이 큽니다. feature extraction 목적이면 jpg도 고려 가능합니다.
IMAGE_FORMAT = "png"  # "png" or "jpg"
JPEG_QUALITY = 90

TCGA_COHORT_PATH = VERIFY_RESULT_PATH / "tcga_survival_cohort_candidate.csv"
CPTAC_COHORT_PATH = VERIFY_RESULT_PATH / "cptac_survival_cohort_candidate.csv"
CPTAC_WSI_SERIES_PATH = CPTAC_RAW_PATH / "matched" / "cptac_pda_matched_wsi_series.csv"

print("PROJECT_DATA_PATH:", PROJECT_DATA_PATH.resolve())
print("IMAGE_DST_PATH:", IMAGE_DST_PATH.resolve())
print("TARGET_MPP:", TARGET_MPP, "TILE_SIZE:", TILE_SIZE)
print("DEBUG_MAX_CASES:", DEBUG_MAX_CASES)



PROJECT_DATA_PATH: /home/user/urbandatalab/YSLee/data/pancreatic_cancer_pathology
IMAGE_DST_PATH: /home/user/urbandatalab/YSLee/data/pancreatic_cancer_pathology/dst/Image
TARGET_MPP: 1.0 TILE_SIZE: 1024
DEBUG_MAX_CASES: None


In [8]:
# TCGA-PAAD SVS WSI tiling: mpp 1.0, 1024 x 1024
# 개선점: low-res tissue mask로 흰 배경 tile 좌표를 먼저 제외하고, case 단위 병렬 처리합니다.

tcga_summary_df = run_tcga_tiling(
    cohort_path=TCGA_COHORT_PATH,
    max_cases=DEBUG_MAX_CASES,
    max_workers=TCGA_MAX_WORKERS,
    target_mpp=TARGET_MPP,
    tile_size=TILE_SIZE,
    tissue_threshold=TISSUE_THRESHOLD,
    prefilter_tissue_threshold=PREFILTER_TISSUE_THRESHOLD,
    tissue_mask_downsample=TCGA_TISSUE_MASK_DOWNSAMPLE,
    skip_existing=SKIP_EXISTING,
    overwrite_metadata=OVERWRITE_METADATA,
    image_format=IMAGE_FORMAT,
    jpeg_quality=JPEG_QUALITY,
    show_tile_progress=SHOW_TILE_PROGRESS,
)

display(tcga_summary_df)
print("saved:", IMAGE_DST_PATH / "TCGA_PAAD" / "tile_summary.csv")


TCGA cases: 100%|██████████| 160/160 [44:01<00:00, 16.51s/case] 


,dataset,case_id,status,wsi_path,native_mpp,read_level,read_level_downsample,read_level_mpp,read_level_tile_size_px,source_tile_size_px,...,n_prefiltered_tiles,n_saved_tiles,case_dir,metadata_path,slide_width,slide_height,mask_level,mask_level_downsample,mask_width,mask_height
0,TCGA_PAAD,TCGA-2J-AAB4,done,../../data/pancreatic_cancer_pathology/raw/TCG...,0.2527,1,4.000060,1.010815,1013,4052,...,264,239,../../data/pancreatic_cancer_pathology/dst/Ima...,../../data/pancreatic_cancer_pathology/dst/Ima...,99599,58920,3,32.004583,3112,1841
1,TCGA_PAAD,TCGA-2J-AAB8,done,../../data/pancreatic_cancer_pathology/raw/TCG...,0.2527,1,4.000155,1.010839,1013,4052,...,237,211,../../data/pancreatic_cancer_pathology/dst/Ima...,../../data/pancreatic_cancer_pathology/dst/Ima...,101591,62563,3,32.004390,3174,1955
2,TCGA_PAAD,TCGA-2J-AAB1,done,../../data/pancreatic_cancer_pathology/raw/TCG...,0.2527,1,4.000075,1.010819,1013,4052,...,314,289,../../data/pancreatic_cancer_pathology/dst/Ima...,../../data/pancreatic_cancer_pathology/dst/Ima...,129479,70141,3,32.007483,4046,2191
3,TCGA_PAAD,TCGA-2J-AAB6,done,../../data/pancreatic_cancer_pathology/raw/TCG...,0.2527,1,4.000127,1.010832,1013,4052,...,374,326,../../data/pancreatic_cancer_pathology/dst/Ima...,../../data/pancreatic_cancer_pathology/dst/Ima...,129479,74739,3,32.004934,4046,2335
4,TCGA_PAAD,TCGA-2J-AABA,done,../../data/pancreatic_cancer_pathology/raw/TCG...,0.2527,1,4.000176,1.010844,1013,4052,...,144,101,../../data/pancreatic_cancer_pathology/dst/Ima...,../../data/pancreatic_cancer_pathology/dst/Ima...,79679,59735,3,32.012390,2489,1866
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,TCGA_PAAD,TCGA-XD-AAUL,done,../../data/pancreatic_cancer_pathology/raw/TCG...,0.2527,1,4.000105,1.010826,1013,4052,...,341,316,../../data/pancreatic_cancer_pathology/dst/Ima...,../../data/pancreatic_cancer_pathology/dst/Ima...,121511,72370,3,32.004902,3797,2261
156,TCGA_PAAD,TCGA-YB-A89D,done,../../data/pancreatic_cancer_pathology/raw/TCG...,0.2527,1,4.000100,1.010825,1013,4052,...,365,310,../../data/pancreatic_cancer_pathology/dst/Ima...,../../data/pancreatic_cancer_pathology/dst/Ima...,111551,85858,3,32.004820,3485,2683
157,TCGA_PAAD,TCGA-YY-A8LH,done,../../data/pancreatic_cancer_pathology/raw/TCG...,0.2527,1,4.000114,1.010829,1013,4052,...,320,266,../../data/pancreatic_cancer_pathology/dst/Ima...,../../data/pancreatic_cancer_pathology/dst/Ima...,93623,80330,3,32.005924,2925,2510
158,TCGA_PAAD,TCGA-Z5-AAPL,done,../../data/pancreatic_cancer_pathology/raw/TCG...,0.2525,1,4.000064,1.010016,1014,4055,...,226,207,../../data/pancreatic_cancer_pathology/dst/Ima...,../../data/pancreatic_cancer_pathology/dst/Ima...,79680,62126,3,32.003606,2490,1941


saved: ../../data/pancreatic_cancer_pathology/dst/Image/TCGA_PAAD/tile_summary.csv


In [9]:
# CPTAC-PDAC DICOM WSI tiling: mpp 1.0, 1024 x 1024
# 개선점: DICOM 전체 pixel_array를 메모리에 올리지 않고 필요한 frame만 부분 디코딩하며, case 단위 병렬 처리합니다.

cptac_summary_df = run_cptac_tiling(
    cohort_path=CPTAC_COHORT_PATH,
    wsi_series_path=CPTAC_WSI_SERIES_PATH,
    max_cases=DEBUG_MAX_CASES,
    max_workers=CPTAC_MAX_WORKERS,
    target_mpp=TARGET_MPP,
    tile_size=TILE_SIZE,
    tissue_threshold=TISSUE_THRESHOLD,
    prefilter_tissue_threshold=PREFILTER_TISSUE_THRESHOLD,
    skip_existing=SKIP_EXISTING,
    overwrite_metadata=OVERWRITE_METADATA,
    image_format=IMAGE_FORMAT,
    jpeg_quality=JPEG_QUALITY,
    show_tile_progress=SHOW_TILE_PROGRESS,
)

display(cptac_summary_df)
print("saved:", IMAGE_DST_PATH / "CPTAC_PDAC" / "tile_summary.csv")


CPTAC cases: 100%|██████████| 140/140 [8:14:07<00:00, 211.77s/case]   


,dataset,case_id,status,SeriesInstanceUID,SeriesDescription,volume_dicom_path,native_mpp,source_tile_size_px,n_saved_tiles,case_dir,metadata_path,n_candidate_tiles,n_prefiltered_tiles,prefilter_mask_mpp,prefilter_mask_path
0,CPTAC_PDAC,C3L-00017,done,1.3.6.1.4.1.5962.99.1.275794962.1770053216.164...,HE tumor,../../data/pancreatic_cancer_pathology/raw/CPT...,0.4942,2072,38,../../data/pancreatic_cancer_pathology/dst/Ima...,../../data/pancreatic_cancer_pathology/dst/Ima...,90,50,1.977098,../../data/pancreatic_cancer_pathology/raw/CPT...
1,CPTAC_PDAC,C3L-00102,done,1.3.6.1.4.1.5962.99.1.278357927.1553878182.164...,HE tumor,../../data/pancreatic_cancer_pathology/raw/CPT...,0.4942,2072,68,../../data/pancreatic_cancer_pathology/dst/Ima...,../../data/pancreatic_cancer_pathology/dst/Ima...,182,85,1.977013,../../data/pancreatic_cancer_pathology/raw/CPT...
2,CPTAC_PDAC,C3L-00277,done,1.3.6.1.4.1.5962.99.1.274184778.1955998775.164...,HE tumor,../../data/pancreatic_cancer_pathology/raw/CPT...,0.4942,2072,73,../../data/pancreatic_cancer_pathology/dst/Ima...,../../data/pancreatic_cancer_pathology/dst/Ima...,160,88,1.977071,../../data/pancreatic_cancer_pathology/raw/CPT...
3,CPTAC_PDAC,C3L-00401,done,1.3.6.1.4.1.5962.99.1.275766711.1523401662.164...,HE tumor,../../data/pancreatic_cancer_pathology/raw/CPT...,0.4942,2072,70,../../data/pancreatic_cancer_pathology/dst/Ima...,../../data/pancreatic_cancer_pathology/dst/Ima...,169,80,1.977013,../../data/pancreatic_cancer_pathology/raw/CPT...
4,CPTAC_PDAC,C3L-00589,done,1.3.6.1.4.1.5962.99.1.279419714.2074340128.164...,HE tumor,../../data/pancreatic_cancer_pathology/raw/CPT...,0.4942,2072,53,../../data/pancreatic_cancer_pathology/dst/Ima...,../../data/pancreatic_cancer_pathology/dst/Ima...,220,65,1.977071,../../data/pancreatic_cancer_pathology/raw/CPT...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,CPTAC_PDAC,C3N-04126,done,1.3.6.1.4.1.5962.99.1.272687026.881903273.1640...,HE tumor,../../data/pancreatic_cancer_pathology/raw/CPT...,0.4942,2072,23,../../data/pancreatic_cancer_pathology/dst/Ima...,../../data/pancreatic_cancer_pathology/dst/Ima...,45,27,1.977098,../../data/pancreatic_cancer_pathology/raw/CPT...
136,CPTAC_PDAC,C3N-04119,done,1.3.6.1.4.1.5962.99.1.276860441.652303345.1640...,HE tumor,../../data/pancreatic_cancer_pathology/raw/CPT...,0.4942,2072,28,../../data/pancreatic_cancer_pathology/dst/Ima...,../../data/pancreatic_cancer_pathology/dst/Ima...,110,38,1.977071,../../data/pancreatic_cancer_pathology/raw/CPT...
137,CPTAC_PDAC,C3N-04282,done,1.3.6.1.4.1.5962.99.1.276827417.1243798057.164...,HE tumor,../../data/pancreatic_cancer_pathology/raw/CPT...,0.5031,2035,43,../../data/pancreatic_cancer_pathology/dst/Ima...,../../data/pancreatic_cancer_pathology/dst/Ima...,161,62,2.012400,../../data/pancreatic_cancer_pathology/raw/CPT...
138,CPTAC_PDAC,C3N-04283,done,1.3.6.1.4.1.5962.99.1.277966612.6938703.164095...,HE tumor,../../data/pancreatic_cancer_pathology/raw/CPT...,0.5031,2035,44,../../data/pancreatic_cancer_pathology/dst/Ima...,../../data/pancreatic_cancer_pathology/dst/Ima...,231,60,2.012400,../../data/pancreatic_cancer_pathology/raw/CPT...


saved: ../../data/pancreatic_cancer_pathology/dst/Image/CPTAC_PDAC/tile_summary.csv


In [ ]:
from pathlib import Path
import json
import re

import pandas as pd

CLINICAL_DST_PATH = PROJECT_DATA_PATH / "dst" / "Clinical"
COMMON_CLINICAL_DIR = COMMON_CLINICAL_RESULT_PATH


def json_safe_value(value):
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        value = value.item()
    return value


def normalize_text(value):
    value = json_safe_value(value)
    if value is None:
        return None
    value = str(value).strip()
    if value == "" or value.lower() in {"nan", "na", "n/a", "unknown", "not reported", "not available"}:
        return None
    return value


def normalize_sex(value):
    value = normalize_text(value)
    if value is None:
        return None
    value = value.lower()
    if value in {"male", "m"}:
        return "male"
    if value in {"female", "f"}:
        return "female"
    return "other_or_unknown"


def normalize_race(value):
    value = normalize_text(value)
    if value is None:
        return None
    value = value.lower().replace("-", " ").strip()
    if value == "white":
        return "white"
    if value in {"black or african american", "black", "african american"}:
        return "black_or_african_american"
    if value == "asian":
        return "asian"
    return "other_or_unknown"


def normalize_vital_status(value):
    value = normalize_text(value)
    if value is None:
        return None
    value = value.lower()
    if value in {"alive", "living"}:
        return "alive"
    if value in {"dead", "deceased"}:
        return "dead"
    return "unknown"


def normalize_stage(value):
    value = normalize_text(value)
    if value is None:
        return None
    value = value.upper().replace("STAGE", "").strip()
    value = re.sub(r"\s+", "", value)
    return f"stage_{value.lower()}" if value else None


def normalize_tnm(value, prefix):
    value = normalize_text(value)
    if value is None:
        return None
    value = value.upper().strip()
    value = re.sub(r"^P", "", value)
    value = re.sub(r"\s+", "", value)
    if not value.startswith(prefix):
        return value
    return value


def normalize_diagnosis(value):
    value = normalize_text(value)
    if value is None:
        return None
    value_lower = value.lower()
    pdac_terms = [
        "pdac",
        "adenocarcinoma",
        "infiltrating duct carcinoma",
        "mucinous adenocarcinoma",
        "adenocarcinoma with mixed subtypes",
    ]
    if any(term in value_lower for term in pdac_terms):
        return "pdac_or_pancreatic_adenocarcinoma"
    if "neuroendocrine" in value_lower:
        return "neuroendocrine_carcinoma"
    return "other_histology"


def standardize_common_clinical(row):
    os_time = json_safe_value(row.get("os_time_days"))
    os_event = json_safe_value(row.get("os_event"))
    age = json_safe_value(row.get("age_years"))
    return {
        "age_years": None if age is None else float(age),
        "sex": normalize_sex(row.get("sex")),
        "race": normalize_race(row.get("race")),
        "vital_status": normalize_vital_status(row.get("vital_status")),
        "os_time_days": None if os_time is None else float(os_time),
        "os_event": None if os_event is None else int(os_event),
        "diagnosis_group": normalize_diagnosis(row.get("diagnosis")),
        "pathologic_stage": normalize_stage(row.get("pathologic_stage")),
        "pathologic_t": normalize_tnm(row.get("pathologic_t"), "T"),
        "pathologic_n": normalize_tnm(row.get("pathologic_n"), "N"),
        "pathologic_m": normalize_tnm(row.get("pathologic_m"), "M"),
        "has_wsi": None if pd.isna(row.get("has_wsi")) else bool(row.get("has_wsi")),
        "has_rnaseq": None if pd.isna(row.get("has_rnaseq")) else bool(row.get("has_rnaseq")),
    }


RAW_CLINICAL_COLUMNS = [
    "age_years", "sex", "race", "vital_status", "os_time_days", "os_event",
    "diagnosis", "pathologic_stage", "pathologic_t", "pathologic_n", "pathologic_m",
    "has_wsi", "has_rnaseq",
]

TCGA_CLINICAL_JSON_DIR = CLINICAL_DST_PATH / "TCGA_PAAD"
TCGA_CLINICAL_JSON_DIR.mkdir(parents=True, exist_ok=True)

tcga_clinical_df = pd.read_csv(COMMON_CLINICAL_DIR / "tcga_common_clinical_harmonized.csv")
tcga_clinical_df = tcga_clinical_df[tcga_clinical_df["cohort"].eq("TCGA_PAAD")].copy()

saved_records = []
for _, row in tcga_clinical_df.iterrows():
    case_id = str(row["subject_id"])
    clinical_standardized = standardize_common_clinical(row)
    clinical_raw = {col: json_safe_value(row.get(col)) for col in RAW_CLINICAL_COLUMNS}

    payload = {
        "dataset": "TCGA_PAAD",
        "case_id": case_id,
        "clinical_schema": "common_tcga_cptac_standardized_v2",
        "clinical": clinical_standardized,
        "clinical_raw": clinical_raw,
        "source": {
            "harmonized_table": (COMMON_CLINICAL_DIR / "tcga_common_clinical_harmonized.csv").as_posix(),
            "variable_map": (COMMON_CLINICAL_DIR / "clinical_common_variable_map.csv").as_posix(),
        },
    }

    out_path = TCGA_CLINICAL_JSON_DIR / f"{case_id}_clinical.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    saved_records.append({"dataset": "TCGA_PAAD", "case_id": case_id, "clinical_json_path": out_path.as_posix()})

tcga_clinical_json_summary = pd.DataFrame(saved_records)
tcga_clinical_json_summary.to_csv(TCGA_CLINICAL_JSON_DIR / "clinical_json_summary.csv", index=False)

print("standardized clinical variables:", list(standardize_common_clinical(tcga_clinical_df.iloc[0]).keys()))
print("saved TCGA clinical JSON:", len(tcga_clinical_json_summary))
print("output_dir:", TCGA_CLINICAL_JSON_DIR)
display(tcga_clinical_json_summary.head())



In [ ]:
from pathlib import Path
import json
import re

import pandas as pd

CLINICAL_DST_PATH = PROJECT_DATA_PATH / "dst" / "Clinical"
COMMON_CLINICAL_DIR = COMMON_CLINICAL_RESULT_PATH


def json_safe_value(value):
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        value = value.item()
    return value


def normalize_text(value):
    value = json_safe_value(value)
    if value is None:
        return None
    value = str(value).strip()
    if value == "" or value.lower() in {"nan", "na", "n/a", "unknown", "not reported", "not available"}:
        return None
    return value


def normalize_sex(value):
    value = normalize_text(value)
    if value is None:
        return None
    value = value.lower()
    if value in {"male", "m"}:
        return "male"
    if value in {"female", "f"}:
        return "female"
    return "other_or_unknown"


def normalize_race(value):
    value = normalize_text(value)
    if value is None:
        return None
    value = value.lower().replace("-", " ").strip()
    if value == "white":
        return "white"
    if value in {"black or african american", "black", "african american"}:
        return "black_or_african_american"
    if value == "asian":
        return "asian"
    return "other_or_unknown"


def normalize_vital_status(value):
    value = normalize_text(value)
    if value is None:
        return None
    value = value.lower()
    if value in {"alive", "living"}:
        return "alive"
    if value in {"dead", "deceased"}:
        return "dead"
    return "unknown"


def normalize_stage(value):
    value = normalize_text(value)
    if value is None:
        return None
    value = value.upper().replace("STAGE", "").strip()
    value = re.sub(r"\s+", "", value)
    return f"stage_{value.lower()}" if value else None


def normalize_tnm(value, prefix):
    value = normalize_text(value)
    if value is None:
        return None
    value = value.upper().strip()
    value = re.sub(r"^P", "", value)
    value = re.sub(r"\s+", "", value)
    if not value.startswith(prefix):
        return value
    return value


def normalize_diagnosis(value):
    value = normalize_text(value)
    if value is None:
        return None
    value_lower = value.lower()
    pdac_terms = [
        "pdac",
        "adenocarcinoma",
        "infiltrating duct carcinoma",
        "mucinous adenocarcinoma",
        "adenocarcinoma with mixed subtypes",
    ]
    if any(term in value_lower for term in pdac_terms):
        return "pdac_or_pancreatic_adenocarcinoma"
    if "neuroendocrine" in value_lower:
        return "neuroendocrine_carcinoma"
    return "other_histology"


def standardize_common_clinical(row):
    os_time = json_safe_value(row.get("os_time_days"))
    os_event = json_safe_value(row.get("os_event"))
    age = json_safe_value(row.get("age_years"))
    return {
        "age_years": None if age is None else float(age),
        "sex": normalize_sex(row.get("sex")),
        "race": normalize_race(row.get("race")),
        "vital_status": normalize_vital_status(row.get("vital_status")),
        "os_time_days": None if os_time is None else float(os_time),
        "os_event": None if os_event is None else int(os_event),
        "diagnosis_group": normalize_diagnosis(row.get("diagnosis")),
        "pathologic_stage": normalize_stage(row.get("pathologic_stage")),
        "pathologic_t": normalize_tnm(row.get("pathologic_t"), "T"),
        "pathologic_n": normalize_tnm(row.get("pathologic_n"), "N"),
        "pathologic_m": normalize_tnm(row.get("pathologic_m"), "M"),
        "has_wsi": None if pd.isna(row.get("has_wsi")) else bool(row.get("has_wsi")),
        "has_rnaseq": None if pd.isna(row.get("has_rnaseq")) else bool(row.get("has_rnaseq")),
    }


RAW_CLINICAL_COLUMNS = [
    "age_years", "sex", "race", "vital_status", "os_time_days", "os_event",
    "diagnosis", "pathologic_stage", "pathologic_t", "pathologic_n", "pathologic_m",
    "has_wsi", "has_rnaseq",
]

CPTAC_CLINICAL_JSON_DIR = CLINICAL_DST_PATH / "CPTAC_PDAC"
CPTAC_CLINICAL_JSON_DIR.mkdir(parents=True, exist_ok=True)

cptac_clinical_df = pd.read_csv(COMMON_CLINICAL_DIR / "cptac_common_clinical_harmonized.csv")
cptac_clinical_df = cptac_clinical_df[cptac_clinical_df["cohort"].eq("CPTAC_PDAC")].copy()

saved_records = []
for _, row in cptac_clinical_df.iterrows():
    case_id = str(row["subject_id"])
    clinical_standardized = standardize_common_clinical(row)
    clinical_raw = {col: json_safe_value(row.get(col)) for col in RAW_CLINICAL_COLUMNS}

    payload = {
        "dataset": "CPTAC_PDAC",
        "case_id": case_id,
        "clinical_schema": "common_tcga_cptac_standardized_v2",
        "clinical": clinical_standardized,
        "clinical_raw": clinical_raw,
        "source": {
            "harmonized_table": (COMMON_CLINICAL_DIR / "cptac_common_clinical_harmonized.csv").as_posix(),
            "variable_map": (COMMON_CLINICAL_DIR / "clinical_common_variable_map.csv").as_posix(),
        },
    }

    out_path = CPTAC_CLINICAL_JSON_DIR / f"{case_id}_clinical.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    saved_records.append({"dataset": "CPTAC_PDAC", "case_id": case_id, "clinical_json_path": out_path.as_posix()})

cptac_clinical_json_summary = pd.DataFrame(saved_records)
cptac_clinical_json_summary.to_csv(CPTAC_CLINICAL_JSON_DIR / "clinical_json_summary.csv", index=False)

print("standardized clinical variables:", list(standardize_common_clinical(cptac_clinical_df.iloc[0]).keys()))
print("saved CPTAC clinical JSON:", len(cptac_clinical_json_summary))
print("output_dir:", CPTAC_CLINICAL_JSON_DIR)
display(cptac_clinical_json_summary.head())



In [ ]:
# TCGA-PAAD RNA-seq 전처리
# raw fpkm_uq_unstranded -> log2(x + 1), 공통 gene 기준 case x gene matrix 저장

from pathlib import Path
import json

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

RNASEQ_DST_PATH = PROJECT_DATA_PATH / "dst" / "RNAseq"
TCGA_RNASEQ_DST = RNASEQ_DST_PATH / "TCGA_PAAD"
TCGA_RNASEQ_DST.mkdir(parents=True, exist_ok=True)

COMMON_CLINICAL_DIR = COMMON_CLINICAL_RESULT_PATH
COMMON_GENE_PATH = COMMON_CLINICAL_DIR / "common_genes_tcga_rna_protein_coding__cptac_rna.csv"
TCGA_RNASEQ_MANIFEST_PATH = TCGA_RAW_PATH / "TCGA_PAAD_matched" / "tcga_paad_matched_rnaseq_files.csv"
TCGA_RNASEQ_RAW_DIR = TCGA_RAW_PATH / "RNA_SEQ_STAR_COUNTS"

common_genes = pd.read_csv(COMMON_GENE_PATH)["gene_symbol"].astype(str).str.upper().tolist()
common_gene_set = set(common_genes)
tcga_rnaseq_manifest = pd.read_csv(TCGA_RNASEQ_MANIFEST_PATH)

# 같은 gene symbol이 여러 row에 있을 경우 protein_coding 우선, 같은 symbol 내 평균으로 collapse합니다.
def load_tcga_expression(file_name: str) -> pd.Series:
    path = TCGA_RNASEQ_RAW_DIR / file_name
    if not path.exists():
        matches = list(TCGA_RNASEQ_RAW_DIR.rglob(file_name))
        if len(matches) == 0:
            raise FileNotFoundError(f"TCGA RNA-seq file not found: {file_name}")
        path = matches[0]
    df = pd.read_csv(path, sep="\t", comment="#")
    df = df[df["gene_name"].notna()].copy()
    df["gene_symbol"] = df["gene_name"].astype(str).str.upper()
    df = df[df["gene_symbol"].isin(common_gene_set)].copy()
    df["expression_log2_fpkm_uq"] = np.log2(pd.to_numeric(df["fpkm_uq_unstranded"], errors="coerce").fillna(0) + 1)
    expr = df.groupby("gene_symbol", sort=False)["expression_log2_fpkm_uq"].mean()
    return expr

expression_records = []
case_ids = []
source_records = []
for _, row in tqdm(tcga_rnaseq_manifest.iterrows(), total=len(tcga_rnaseq_manifest), desc="TCGA RNA-seq"):
    case_id = row["patient_id"]
    expr = load_tcga_expression(row["file_name"])
    expression_records.append(expr.reindex(common_genes))
    case_ids.append(case_id)
    source_records.append(
        {
            "dataset": "TCGA_PAAD",
            "case_id": case_id,
            "sample_id": row["sample_id"],
            "sample_type": row["sample_type"],
            "file_id": row["file_id"],
            "file_name": row["file_name"],
            "source_value_column": "fpkm_uq_unstranded",
            "expression_transform": "log2(fpkm_uq_unstranded + 1)",
        }
    )

tcga_expr_log2 = pd.DataFrame(expression_records, index=case_ids, columns=common_genes)
tcga_expr_log2.index.name = "case_id"

tcga_gene_mean = tcga_expr_log2.mean(axis=0, skipna=True)
tcga_gene_std = tcga_expr_log2.std(axis=0, skipna=True).replace(0, np.nan)
tcga_expr_zscore = (tcga_expr_log2 - tcga_gene_mean) / tcga_gene_std
tcga_expr_zscore = tcga_expr_zscore.fillna(0)

gene_df = pd.DataFrame({"gene_symbol": common_genes})
gene_df.to_csv(TCGA_RNASEQ_DST / "genes_common_protein_coding.csv", index=False)
tcga_expr_log2.to_csv(TCGA_RNASEQ_DST / "matrix_common_genes_log2_fpkm_uq.csv")
tcga_expr_zscore.to_csv(TCGA_RNASEQ_DST / "matrix_common_genes_zscore.csv")
pd.DataFrame(source_records).to_csv(TCGA_RNASEQ_DST / "rnaseq_source_files.csv", index=False)
pd.DataFrame({"gene_symbol": common_genes, "mean_log2": tcga_gene_mean.values, "std_log2": tcga_gene_std.values}).to_csv(
    TCGA_RNASEQ_DST / "zscore_reference.csv", index=False
)

case_json_records = []
for case_id in tqdm(tcga_expr_zscore.index, desc="TCGA RNA-seq case files"):
    feature_path = TCGA_RNASEQ_DST / f"{case_id}_rnaseq_zscore.npy"
    np.save(feature_path, tcga_expr_zscore.loc[case_id].to_numpy(dtype=np.float32))
    payload = {
        "dataset": "TCGA_PAAD",
        "case_id": case_id,
        "rnaseq_schema": "common_tcga_cptac_rnaseq_v1",
        "n_genes": len(common_genes),
        "gene_file": (TCGA_RNASEQ_DST / "genes_common_protein_coding.csv").as_posix(),
        "feature_file": feature_path.as_posix(),
        "matrix_log2_file": (TCGA_RNASEQ_DST / "matrix_common_genes_log2_fpkm_uq.csv").as_posix(),
        "matrix_zscore_file": (TCGA_RNASEQ_DST / "matrix_common_genes_zscore.csv").as_posix(),
        "expression_raw_unit": "fpkm_uq_unstranded",
        "expression_log2_unit": "log2(fpkm_uq_unstranded + 1)",
        "feature_unit": "gene_zscore_within_TCGA_PAAD",
    }
    json_path = TCGA_RNASEQ_DST / f"{case_id}_rnaseq.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    case_json_records.append({"dataset": "TCGA_PAAD", "case_id": case_id, "rnaseq_json_path": json_path.as_posix(), "feature_path": feature_path.as_posix()})

rnaseq_json_summary = pd.DataFrame(case_json_records)
rnaseq_json_summary.to_csv(TCGA_RNASEQ_DST / "rnaseq_json_summary.csv", index=False)

print("TCGA expression log2 matrix:", tcga_expr_log2.shape)
print("TCGA expression zscore matrix:", tcga_expr_zscore.shape)
print("n missing values log2:", int(tcga_expr_log2.isna().sum().sum()))
print("output_dir:", TCGA_RNASEQ_DST)
display(rnaseq_json_summary.head())



In [ ]:
# CPTAC-PDAC RNA-seq 전처리
# raw rna_tumor_rsem_uq_log2.cct -> 공통 gene 기준 case x gene matrix 저장

from pathlib import Path
import json

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

RNASEQ_DST_PATH = PROJECT_DATA_PATH / "dst" / "RNAseq"
CPTAC_RNASEQ_DST = RNASEQ_DST_PATH / "CPTAC_PDAC"
CPTAC_RNASEQ_DST.mkdir(parents=True, exist_ok=True)

COMMON_CLINICAL_DIR = COMMON_CLINICAL_RESULT_PATH
COMMON_GENE_PATH = COMMON_CLINICAL_DIR / "common_genes_tcga_rna_protein_coding__cptac_rna.csv"
CPTAC_RNASEQ_RAW_PATH = CPTAC_RAW_PATH / "omics" / "rna_tumor_rsem_uq_log2.cct"

common_genes = pd.read_csv(COMMON_GENE_PATH)["gene_symbol"].astype(str).str.upper().tolist()
common_gene_set = set(common_genes)

cptac_raw = pd.read_csv(CPTAC_RNASEQ_RAW_PATH, sep="\t")
gene_col = cptac_raw.columns[0]
cptac_raw["gene_symbol"] = cptac_raw[gene_col].astype(str).str.upper()
cptac_raw = cptac_raw[cptac_raw["gene_symbol"].isin(common_gene_set)].copy()

sample_cols = [c for c in cptac_raw.columns if c not in [gene_col, "gene_symbol"]]
expr_gene_by_case = cptac_raw.groupby("gene_symbol", sort=False)[sample_cols].mean()
cptac_expr_log2 = expr_gene_by_case.reindex(common_genes).T
cptac_expr_log2.index.name = "case_id"

cptac_gene_mean = cptac_expr_log2.mean(axis=0, skipna=True)
cptac_gene_std = cptac_expr_log2.std(axis=0, skipna=True).replace(0, np.nan)
cptac_expr_zscore = (cptac_expr_log2 - cptac_gene_mean) / cptac_gene_std
cptac_expr_zscore = cptac_expr_zscore.fillna(0)

gene_df = pd.DataFrame({"gene_symbol": common_genes})
gene_df.to_csv(CPTAC_RNASEQ_DST / "genes_common_protein_coding.csv", index=False)
cptac_expr_log2.to_csv(CPTAC_RNASEQ_DST / "matrix_common_genes_log2_rsem_uq.csv")
cptac_expr_zscore.to_csv(CPTAC_RNASEQ_DST / "matrix_common_genes_zscore.csv")
pd.DataFrame({"gene_symbol": common_genes, "mean_log2": cptac_gene_mean.values, "std_log2": cptac_gene_std.values}).to_csv(
    CPTAC_RNASEQ_DST / "zscore_reference.csv", index=False
)

case_json_records = []
for case_id in tqdm(cptac_expr_zscore.index, desc="CPTAC RNA-seq case files"):
    feature_path = CPTAC_RNASEQ_DST / f"{case_id}_rnaseq_zscore.npy"
    np.save(feature_path, cptac_expr_zscore.loc[case_id].to_numpy(dtype=np.float32))
    payload = {
        "dataset": "CPTAC_PDAC",
        "case_id": case_id,
        "rnaseq_schema": "common_tcga_cptac_rnaseq_v1",
        "n_genes": len(common_genes),
        "gene_file": (CPTAC_RNASEQ_DST / "genes_common_protein_coding.csv").as_posix(),
        "feature_file": feature_path.as_posix(),
        "matrix_log2_file": (CPTAC_RNASEQ_DST / "matrix_common_genes_log2_rsem_uq.csv").as_posix(),
        "matrix_zscore_file": (CPTAC_RNASEQ_DST / "matrix_common_genes_zscore.csv").as_posix(),
        "expression_raw_unit": "rsem_uq_log2",
        "expression_log2_unit": "rsem_uq_log2 as provided",
        "feature_unit": "gene_zscore_within_CPTAC_PDAC",
    }
    json_path = CPTAC_RNASEQ_DST / f"{case_id}_rnaseq.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    case_json_records.append({"dataset": "CPTAC_PDAC", "case_id": case_id, "rnaseq_json_path": json_path.as_posix(), "feature_path": feature_path.as_posix()})

rnaseq_json_summary = pd.DataFrame(case_json_records)
rnaseq_json_summary.to_csv(CPTAC_RNASEQ_DST / "rnaseq_json_summary.csv", index=False)

print("CPTAC expression log2 matrix:", cptac_expr_log2.shape)
print("CPTAC expression zscore matrix:", cptac_expr_zscore.shape)
print("n missing values log2:", int(cptac_expr_log2.isna().sum().sum()))
print("output_dir:", CPTAC_RNASEQ_DST)
display(rnaseq_json_summary.head())



### 7. Survival-associated RNA-seq feature selection for M3/M4

M3/M4 모델 입력 차원을 줄이기 위해 PDAC 문헌 기반 curated gene set을 먼저 구성하고 중복 gene을 제거한다. 이후 train split에서 RNA-seq gene별 univariate Cox score test를 수행하여 TCGA-PAAD와 CPTAC-PDAC의 Cox z-score를 Stouffer 방식으로 meta-analysis한다. 최종 literature-guided set은 문헌 curated gene을 우선 포함하고, 부족한 feature 수는 train split Cox ranking으로 채운다. Validation/test label은 gene selection에 사용하지 않으며, 최종 저장 feature는 기존 z-score 값을 선택 gene 순서로 잘라 사용한다. 기본 후보는 1,000개, 1,500개, 2,000개이며, 모델 학습에서는 literature-guided 1,500개를 기본값으로 사용하고 나머지는 ablation으로 비교한다.


In [ ]:
# M3/M4용 RNA-seq feature selection
# 18,879 common genes -> PDAC literature-guided + train-split survival Cox top 1,000 / 1,500 / 2,000 genes

from scripts.select_rnaseq_gene_features import (
    FEATURE_SELECTION_RESULT_PATH,
    SELECTED_RNASEQ_DST_PATH,
    run_feature_selection,
)

RNASEQ_SELECTED_GENE_COUNTS = [1000, 1500, 2000]
ranked_genes, rnaseq_selection_summary, rnaseq_selection_split = run_feature_selection(RNASEQ_SELECTED_GENE_COUNTS)

print("ranked_genes:", ranked_genes.shape)
print("selected feature output:", SELECTED_RNASEQ_DST_PATH)
print("selection summary output:", FEATURE_SELECTION_RESULT_PATH)
display(rnaseq_selection_summary)
display(rnaseq_selection_split.groupby(["split", "dataset"])["case_id"].count().unstack(fill_value=0))
print("literature-guided top genes")
display(ranked_genes.head(20))
